# Curating systems for SLiM simulations

**Useful commands**

Submitting many slurm scripts
* In system dir: ``for d in */; do (cd "$d" && sbatch gpu.sh); done``
* In sampling dir: ``for d in */; do (cd "$d" && (for s in */; do (cd "$s" && sbatch gpu.sh); done)); done``

Submitting slurm scripts where no ``kd.csv`` files exist
* In system dir: ``for d in */; do (cd "$d" && [ ! -f kd.csv ] && sbatch gpu.sh); done``
* In sampling dir: ``for d in */; do (cd "$d" && (for s in */; do (cd "$s" && [ ! -f kd.csv ] && sbatch gpu.sh); done)); done``

Manual SMOG run
* PDBAdjust: ``~/software/smog-2.5/bin/smog_adjustPDB -i $raw_pdb -o $adjusted_pdb -removewater -renumber -d``

Checking number of frames
* From simulation dir: ``head -c 12 *.dcd | tail -c 4 | od -An -t dI``

In [1]:
import json
import os
import pandas as pd
import numpy as np
import subprocess

while os.path.split(os.getcwd())[-1] != 'calvados-slim':
    os.chdir('..')

from src import curation, utils

/groups/sbinlab/fknudsen/software/miniconda3/envs/calvados/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Curating simulations

In [2]:
# Loading curations
with open('data/curation/systems.json') as file:
    systems = json.load(file)
systems_dir = os.path.abspath('data/curation/systems')

# Loading configurations
with open('data/configurations/conditions.json') as file:
    conditions = json.load(file)
with open('data/configurations/sampling.json') as file:
    sampling = json.load(file)
with open('data/configurations/go_scans.json') as file:
    go_scans = json.load(file)
slurm_scripts_dir = 'data/configurations/slurm'

In [3]:
# Setting modes to create runs for
conditions_mode = 'Default'
sampling_mode = 'Scan'

scan_dir = os.path.abspath(f"sim/{sampling_mode.lower()}")

In [12]:
# Defining k range
k_min  = 0
k_max  = 5
k_step = 0.1
ks = np.arange(k_min, k_max+k_step, k_step)
ks

array([0. , 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. , 1.1, 1.2,
       1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2. , 2.1, 2.2, 2.3, 2.4, 2.5,
       2.6, 2.7, 2.8, 2.9, 3. , 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8,
       3.9, 4. , 4.1, 4.2, 4.3, 4.4, 4.5, 4.6, 4.7, 4.8, 4.9, 5. ])

In [ ]:
# Generating simulation setups

# Using date as hash
hash = pd.Timestamp.now('utc').strftime("%Y%m%d")

platform = input('Which platform [CPU/GPU]?').upper()

# Looping over systems
for system in systems:

    system_dir = os.path.join(systems_dir, system['ID'])

    # Looping over Go k scan
    for k in ks:

        # Creating simulation directory
        sim_dir = os.path.join(scan_dir, system['ID'], f"k={k:.2f}")
        if os.path.exists(sim_dir):
            continue

        print(f"Creating '{sim_dir}'")
        curation.Simulation.build(
            system,
            system_dir,
            conditions[conditions_mode],
            sampling[sampling_mode],
            k, sim_dir, platform
        )



## Inspecting and submitting simulations

In [29]:
# Inspecting simulations
sims = []
for system in os.listdir(scan_dir):
    system_dir = os.path.join(scan_dir, system)
    for k_dir in os.listdir(system_dir):
        k = float(k_dir.split('=')[-1])
        sim_dir = os.path.join(system_dir, k_dir)
        sims.append({
            'System': system,
            'k': k,
            'Frames': utils.get_dcd_frame_count(os.path.join(sim_dir, f'{system}.dcd')),
            'Path': sim_dir,
        })
sims = pd.DataFrame(sims).sort_values(['System','k'])

In [32]:
# Selecting simulations for submission

# Criteria options
not_simulated = lambda row: np.isnan(row['Frames'])
running = lambda row: row['Frames'] < 10000
whole_k = lambda row: row['k'] % 1 == 0
specific_ks = lambda ks: lambda row: row['k'] in [round(f, 1) for f in ks]
specific_system = lambda sys: lambda row: row['System'] == sys

criteria = [
    specific_ks(np.arange(1,3.1,0.1)),
    specific_system('klhdc2-ephb2'),
    not_simulated
] 

sims_sel = sims.copy()
for criterion in criteria:
    sims_sel = sims_sel[sims_sel.apply(criterion, axis=1)]
sims_sel

,System,k,Frames,Path
373,klhdc2-ephb2,1.1,NaN,/home/fknudsen/calvados-slim/sim/scan/klhdc2-e...
395,klhdc2-ephb2,1.2,NaN,/home/fknudsen/calvados-slim/sim/scan/klhdc2-e...
404,klhdc2-ephb2,1.3,NaN,/home/fknudsen/calvados-slim/sim/scan/klhdc2-e...
400,klhdc2-ephb2,1.4,NaN,/home/fknudsen/calvados-slim/sim/scan/klhdc2-e...
389,klhdc2-ephb2,1.5,NaN,/home/fknudsen/calvados-slim/sim/scan/klhdc2-e...
385,klhdc2-ephb2,1.6,NaN,/home/fknudsen/calvados-slim/sim/scan/klhdc2-e...
366,klhdc2-ephb2,1.7,NaN,/home/fknudsen/calvados-slim/sim/scan/klhdc2-e...
357,klhdc2-ephb2,1.8,NaN,/home/fknudsen/calvados-slim/sim/scan/klhdc2-e...
376,klhdc2-ephb2,1.9,NaN,/home/fknudsen/calvados-slim/sim/scan/klhdc2-e...
379,klhdc2-ephb2,2.1,NaN,/home/fknudsen/calvados-slim/sim/scan/klhdc2-e...


In [ ]:
# Submitting simulations
for _, sim in sims_sel.iterrows():
    break # COMMENT AWAY MANUALLY - SUBMIT WITH CARE
    subprocess.run(
        ['sbatch', 'robust_gpu.sh'],
        cwd=sim['Path']
    )

Submitted batch job 5539006
Submitted batch job 5539007
Submitted batch job 5539008
Submitted batch job 5539009
Submitted batch job 5539010
Submitted batch job 5539011
Submitted batch job 5539012
Submitted batch job 5539013
Submitted batch job 5539014
Submitted batch job 5539015
Submitted batch job 5539016
Submitted batch job 5539017
Submitted batch job 5539018
Submitted batch job 5539019
Submitted batch job 5539020
Submitted batch job 5539021
Submitted batch job 5539022
Submitted batch job 5539023
